<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/NEW_BENCHMARK_TOPOLOGICAL_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AST_LEFM standing for Arithmetic Spectral Theory - Laplace Euler Fourier Mellin.

These pieces combine into the L-EFM operator tested in the notebook, it acts as a rigid mathematical filter. It explains why the "Spectral Trap" drastically rejects values outside of $\sigma = 0.5$ due to massive error amplification, while establishing the hard boundary constant ($\Lambda$) used for the geometric governance and AI safety vector filtering.It is a beautiful synthesis of classical analytic number theory and modern operator physics.

AST/L-EFM - A Complete Python Library for Spectral Quantification of Prime Theorems, Proof of the Riemann Hypothesis, and Deterministic AI Safety: https://zenodo.org/records/20275803

In [1]:
!nvidia-smi

Wed Jul  8 22:00:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## SETUP

In [2]:
!git clone https://github.com/frank-morales2020/ast_lefm.git

Cloning into 'ast_lefm'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 43 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 20.91 KiB | 2.99 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [3]:
!ls -ltha /content/ast_lefm/

total 60K
drwxr-xr-x 4 root root 4.0K Jul  8 22:00 .
-rw-r--r-- 1 root root  970 Jul  8 22:00 chowla.py
-rw-r--r-- 1 root root 1.1K Jul  8 22:00 constants.py
-rw-r--r-- 1 root root 2.6K Jul  8 22:00 einstein.py
drwxr-xr-x 8 root root 4.0K Jul  8 22:00 .git
-rw-r--r-- 1 root root 1.5K Jul  8 22:00 gtt_decay.py
-rw-r--r-- 1 root root 2.0K Jul  8 22:00 h2e.py
-rw-r--r-- 1 root root 1.1K Jul  8 22:00 __init__.py
-rw-r--r-- 1 root root 3.2K Jul  8 22:00 lefm.py
-rw-r--r-- 1 root root 5.3K Jul  8 22:00 README.md
-rw-r--r-- 1 root root  911 Jul  8 22:00 setup.py
-rw-r--r-- 1 root root 1.9K Jul  8 22:00 sieve.py
drwxr-xr-x 2 root root 4.0K Jul  8 22:00 test
drwxr-xr-x 1 root root 4.0K Jul  8 22:00 ..


In [4]:
!ls -ltha /content/ast_lefm/test/

total 12K
drwxr-xr-x 2 root root 4.0K Jul  8 22:00 .
drwxr-xr-x 4 root root 4.0K Jul  8 22:00 ..
-rw-r--r-- 1 root root 2.2K Jul  8 22:00 test_all.py


In [5]:
%cd /content/ast_lefm
!pip install -e .

/content/ast_lefm
Obtaining file:///content/ast_lefm
  Preparing metadata (setup.py) ... done
  Running setup.py develop for ast_lefm


In [6]:
import sys
sys.path.insert(0, '/content/ast_lefm')

In [7]:
import sys
import os

# Force add the path
sys.path.insert(0, '/content/ast_lefm')
sys.path.insert(0, '/content/ast_lefm/ast_lefm')

# Try to import
try:
    from ast_lefm.sieve import primes_up_to
    print("✓ Import successful")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    # Let's see what's in the directory
    print("\nFiles in /content/ast_lefm:")
    os.listdir('/content/ast_lefm')

    print("\nTrying alternative import...")
    # Try importing as a local module
    import importlib.util
    spec = importlib.util.spec_from_file_location("sieve", "/content/ast_lefm/sieve.py")
    sieve = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(sieve)
    primes_up_to = sieve.primes_up_to
    print("✓ Alternative import worked")

✓ Import successful


In [8]:
!pip install --upgrade bitsandbytes accelerate transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 228.3 MB/s eta 0:00:00


In [9]:
import torch
import bitsandbytes as bnb
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'bitsandbytes: {bnb.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.11.0+cu128
CUDA: 12.8
bitsandbytes: 0.49.2
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## TWO-TASK BENCHMARK (3 EPOCHS)

In [18]:
# ============================================================================
# TWO-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline
#
# Task A — World vs Sports       (classes 0, 1)
# Task B — Business vs Sci/Tech  (classes 2, 3)
#
# Key fixes vs original notebook:
# 1. Separate classifiers per task (no shared head artifact)
# 2. Proper anchor handling for Topological (zero gradients + enforce)
# 3. Correct EWC implementation with Fisher diagonal
# 4. Proper Replay with task-specific heads
# 5. Honest HOPE-like with dual-timescale EMA
# 6. Fair comparison with identical architecture across all methods
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import random
import pandas as pd
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from sklearn.metrics import accuracy_score
from warnings import filterwarnings
filterwarnings('ignore')

import sys
sys.path.insert(0, '/content/ast_lefm')
try:
    from ast_lefm.sieve import primes_up_to
except ImportError:
    def primes_up_to(n):
        """Fallback sieve of Eratosthenes."""
        sieve = [True] * (n + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(n ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, n + 1, i):
                    sieve[j] = False
        return [i for i in range(2, n + 1) if sieve[i]]


# ============================================================
# GLOBAL CONFIGURATION
# ============================================================
SEED         = 123
N_RUNS       = 3       # Mean ± std over runs
BATCH_SIZE   = 16
REPLAY_HALF  = 8
MAX_LEN      = 64
EPOCHS       = 3
LR_EMBED     = 5e-3
LR_CLS       = 1e-3
EWC_LAMBDA   = 5000
EMA_BETA     = 0.9
EMA_LAMBDA   = 5000
PRIME_LIMIT  = 13
BUFFER_SIZE  = 200
MODEL_NAME   = "openai/gpt-oss-20b"


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("=" * 80)
print("TWO-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline")
print("=" * 80)


# =====================================================================
# 1. DATASET LOADING (Parquet download only)
# =====================================================================
print("\nLoading dataset from parquet...")

try:
    parquet_url = "https://huggingface.co/datasets/ag_news/resolve/main/data/train-00000-of-00001.parquet"
    df = pd.read_parquet(parquet_url)
    dataset = Dataset.from_pandas(df)
    print(f"✓ Success! Dataset loaded with {len(dataset)} samples")
except Exception as e:
    print(f"✗ Failed to load dataset: {e}")
    raise RuntimeError("Could not load dataset from parquet URL")

def create_task(dataset, class_labels, num_samples=500):
    """Create a task with specific class labels."""
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    actual_samples = min(num_samples, len(filtered))
    sampled = filtered.select(range(actual_samples))
    texts = [str(item['text']) for item in sampled]
    labels = [int(item['label']) % 2 for item in sampled]
    return texts, labels

# Task A: World (0) vs Sports (1)
task_a_texts, task_a_labels = create_task(dataset, [0, 1], 500)

# Task B: Business (2) vs Sci/Tech (3)
task_b_texts, task_b_labels = create_task(dataset, [2, 3], 1000)

print(f"Task A: {len(task_a_texts)} samples  (World=0, Sports=1)")
print(f"Task B: {len(task_b_texts)} samples  (Business=0, Sci/Tech=1)")


# =====================================================================
# 2. MODEL — SEPARATE CLASSIFIER PER TASK
# =====================================================================
class TaskAwareModel(nn.Module):
    """
    Separate classifier heads ensure fair comparison.
    Only the shared embedding layer can cause cross-task interference.
    """
    def __init__(self, base_model, hidden_size=2880):
        super().__init__()
        self.base_model   = base_model
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        last_hidden = outputs.hidden_states[-1][:, -1, :]
        head = self.classifier_A if self.current_task == 'A' else self.classifier_B
        return head(last_hidden), outputs

    def switch_task(self, task: str):
        assert task in ('A', 'B'), f"Unknown task: {task}"
        self.current_task = task

    def freeze_classifier_a(self):
        """Freeze Task A head after training."""
        self.classifier_A.requires_grad_(False)


# =====================================================================
# 3. SHARED UTILITIES
# =====================================================================
def get_embed_layer(base_model: nn.Module) -> nn.Embedding:
    """Locate the token embedding layer."""
    if hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'wte'):
        return base_model.transformer.wte
    for module in base_model.modules():
        if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100_000:
            return module
    raise RuntimeError("Could not locate embedding layer.")


@torch.no_grad()
def evaluate(model, tokenizer, texts, labels, task: str, batch_size: int = 32) -> float:
    """Pure evaluation — no side effects."""
    was_training = model.training
    previous_task = model.current_task
    model.eval()
    model.switch_task(task)

    all_preds, all_labels = [], []
    for i in range(0, len(texts), batch_size):
        tokens = tokenizer(
            texts[i:i + batch_size], return_tensors="pt",
            padding=True, truncation=True, max_length=MAX_LEN,
        ).to(device)
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels[i:i + batch_size])

    model.switch_task(previous_task)
    if was_training:
        model.train()
    return accuracy_score(all_labels, all_preds)


def tokenize(tokenizer, texts, labels):
    """Tokenize texts and convert labels to tensors."""
    tokens = tokenizer(
        texts, return_tensors="pt",
        padding=True, truncation=True, max_length=MAX_LEN,
    ).to(device)
    return tokens, torch.tensor(labels, dtype=torch.long).to(device)


def load_fresh_model():
    """Load clean model copy. Backbone is immediately frozen."""
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = TaskAwareModel(base_model).to(device)
    for param in base_model.parameters():
        param.requires_grad = False
    return model, tokenizer, base_model


def train_task_a(model, tokenizer, optimizer):
    """Shared Task A training loop."""
    model.switch_task('A')
    model.train()
    for _ in range(EPOCHS):
        for i in range(0, len(task_a_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_a_texts[i:i + BATCH_SIZE],
                task_a_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            F.cross_entropy(logits, labels).backward()
            optimizer.step()


def build_task_b_optimizer(embed_layer, model):
    """Task B optimizer: embedding + classifier_B only."""
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight, 'lr': LR_EMBED},
        {'params': model.classifier_B.parameters(), 'lr': LR_CLS},
    ])


def task_b_step(loss, embed_layer, optimizer, max_norm=1.0):
    """Shared Task B gradient step with clipping."""
    loss.backward()
    torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=max_norm)
    optimizer.step()


def summarise(results: list) -> dict:
    """Summarize results with mean ± std."""
    fgt = [r['forgetting'] for r in results]
    ab = [r['acc_b'] for r in results]
    aa = [r['acc_a_final'] for r in results]
    return {
        'forgetting_mean': np.mean(fgt), 'forgetting_std': np.std(fgt),
        'acc_b_mean': np.mean(ab), 'acc_b_std': np.std(ab),
        'acc_a_final_mean': np.mean(aa), 'acc_a_final_std': np.std(aa),
    }


def log_run(acc_a_initial, acc_a_final, acc_b) -> float:
    """Log results for a single run."""
    forgetting = (acc_a_initial - acc_a_final) * 100
    print(f"    Task A initial: {acc_a_initial * 100:.1f}%")
    print(f"    Task A final:   {acc_a_final * 100:.1f}%  |  Forgetting: {forgetting:+.1f}%")
    print(f"    Task B acc:     {acc_b * 100:.1f}%")
    return forgetting


def cleanup(*objects):
    """Clean up memory."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


# =====================================================================
# 4. TOPOLOGICAL GOVERNOR
# =====================================================================
class TopologicalGovernor:
    """
    Anchors prime-indexed embedding rows after Task A.
    Cost: FLAT — fixed number of rows regardless of task count.
    """
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer = embed_layer
        vocab_size = embed_layer.weight.shape[0]
        self.anchor_indices = [p for p in primes_up_to(prime_limit) if p < vocab_size]
        self.snapshot = {}
        print(f"  → Anchoring {len(self.anchor_indices)} prime rows: {self.anchor_indices}")

    def take_snapshot(self):
        """Snapshot anchor rows immediately after Task A."""
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def enforce_anchors(self):
        """Restore anchor rows after each optimizer step."""
        if not self.snapshot:
            raise RuntimeError("Call take_snapshot() before enforce_anchors().")
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    @torch.no_grad()
    def zero_anchor_gradients(self):
        """Zero gradients for anchor rows before optimizer step."""
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        """Verify anchor rows haven't drifted."""
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )


# =====================================================================
# 5. FISHER DIAGONAL (for EWC)
# =====================================================================
def compute_fisher(model, tokenizer, embed_layer, n_samples: int = 200) -> torch.Tensor:
    """
    Empirical Fisher Information diagonal on embedding weight matrix.
    Estimated on Task A data immediately after Task A training.
    """
    model.eval()
    model.switch_task('A')
    fisher = torch.zeros_like(embed_layer.weight, dtype=torch.float32)
    n_batches = 0

    for i in range(0, min(n_samples, len(task_a_texts)), BATCH_SIZE):
        tokens, labels = tokenize(
            tokenizer,
            task_a_texts[i:i + BATCH_SIZE],
            task_a_labels[i:i + BATCH_SIZE],
        )
        model.zero_grad()
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        F.cross_entropy(logits, labels).backward()
        if embed_layer.weight.grad is not None:
            fisher += embed_layer.weight.grad.float() ** 2
            n_batches += 1

    fisher /= max(n_batches, 1)
    model.zero_grad()
    return fisher


# =====================================================================
# 6. DUAL TIMESCALE EMA (for HOPE-like)
# =====================================================================
class DualTimescaleEMA:
    """
    Dual-timescale EMA: slow tracker + penalty pulls fast weights toward it.
    Cost: FLAT — one EMA matrix regardless of task count.
    """
    def __init__(self, embed_layer: nn.Embedding,
                 beta: float = EMA_BETA, lambda_penalty: float = EMA_LAMBDA):
        self.embed_layer = embed_layer
        self.beta = beta
        self.lambda_penalty = lambda_penalty
        self.slow_embed = embed_layer.weight.detach().clone().float()

    @torch.no_grad()
    def update_slow(self):
        """EMA update after each optimizer step."""
        current = self.embed_layer.weight.float()
        self.slow_embed = self.beta * self.slow_embed + (1.0 - self.beta) * current

    def penalty(self) -> torch.Tensor:
        """L2 penalty pulling fast weights toward slow EMA."""
        delta = self.embed_layer.weight.float() - self.slow_embed
        return self.lambda_penalty * (delta ** 2).sum()


# =====================================================================
# 7. BASELINE METHOD (No protection)
# =====================================================================
def train_baseline():
    """Baseline: shared embedding, no protection."""
    print(f"\n{'=' * 60}")
    print("Method: BASELINE  (shared embedding, no protection)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        # Train Task A
        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight, 'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(), 'lr': LR_CLS},
            {'params': model.classifier_B.parameters(), 'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        # Train Task B — embedding drifts freely
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_b_step(F.cross_entropy(logits, labels), embed_layer, opt_b)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 8. TOPOLOGICAL AI METHOD
# =====================================================================
def train_topological():
    """Topological AI: anchor prime embedding rows."""
    print(f"\n{'=' * 60}")
    print("Method: TOPOLOGICAL AI  (prime embedding anchors)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        # Train Task A
        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight, 'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(), 'lr': LR_CLS},
            {'params': model.classifier_B.parameters(), 'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # Snapshot anchors after Task A
        governor = TopologicalGovernor(embed_layer)
        governor.take_snapshot()
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        # Train Task B with anchor protection
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()  # Zero anchor gradients
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
                opt_b.step()
                governor.enforce_anchors()        # Restore anchors

        assert governor.verify_integrity(), "Anchor integrity check failed!"

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 9. EWC METHOD (Elastic Weight Consolidation)
# =====================================================================
def train_ewc():
    """EWC: Fisher-weighted penalty on embedding drift."""
    print(f"\n{'=' * 60}")
    print("Method: EWC  (Fisher Information diagonal on embedding)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        # Train Task A
        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight, 'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(), 'lr': LR_CLS},
            {'params': model.classifier_B.parameters(), 'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # Compute Fisher and snapshot embedding
        fisher = compute_fisher(model, tokenizer, embed_layer)
        embed_snapshot = embed_layer.weight.detach().clone().float()
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        # Train Task B with EWC penalty
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss = F.cross_entropy(logits, labels)
                delta = embed_layer.weight.float() - embed_snapshot
                ewc_loss = (fisher * delta ** 2).sum()
                task_b_step(task_loss + EWC_LAMBDA * ewc_loss, embed_layer, opt_b)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model, fisher, embed_snapshot)

    return summarise(results)


# =====================================================================
# 10. REPLAY METHOD
# =====================================================================
def train_replay():
    """Replay: store Task A samples and replay during Task B training."""
    print(f"\n{'=' * 60}")
    print("Method: REPLAY  (Task A replay via classifier_A)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        # Train Task A
        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight, 'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(), 'lr': LR_CLS},
            {'params': model.classifier_B.parameters(), 'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # Store Task A samples in buffer
        buffer = list(zip(task_a_texts, task_a_labels))[:BUFFER_SIZE]
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        # Train Task B with replay
        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                opt_b.zero_grad()

                # Task B forward pass
                model.switch_task('B')
                tokens_b, labels_b = tokenize(
                    tokenizer,
                    task_b_texts[i:i + REPLAY_HALF],
                    task_b_labels[i:i + REPLAY_HALF],
                )
                logits_b, _ = model(tokens_b.input_ids, tokens_b.attention_mask)
                loss_b = F.cross_entropy(logits_b, labels_b)

                # Task A replay via classifier_A
                replay_batch = random.sample(buffer, min(REPLAY_HALF, len(buffer)))
                r_texts, r_labels = zip(*replay_batch)
                tokens_a, labels_a = tokenize(tokenizer, list(r_texts), list(r_labels))
                model.switch_task('A')
                logits_a, _ = model(tokens_a.input_ids, tokens_a.attention_mask)
                loss_a = F.cross_entropy(logits_a, labels_a)

                # Combined backward: both tasks pull shared embedding
                task_b_step(loss_b + loss_a, embed_layer, opt_b)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 11. HOPE-LIKE METHOD (Dual Timescale EMA)
# =====================================================================
def train_hope():
    """HOPE-like: dual-timescale EMA penalty on embedding."""
    print(f"\n{'=' * 60}")
    print("Method: HOPE-like  (Dual Timescale EMA on embedding)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        # Train Task A
        opt_a = bnb.optim.AdamW8bit([
            {'params': embed_layer.weight, 'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(), 'lr': LR_CLS},
            {'params': model.classifier_B.parameters(), 'lr': LR_CLS},
        ])
        train_task_a(model, tokenizer, opt_a)

        # Initialize EMA after Task A
        ema = DualTimescaleEMA(embed_layer, beta=EMA_BETA, lambda_penalty=EMA_LAMBDA)
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')

        # Train Task B with EMA penalty
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)

        for _ in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss = F.cross_entropy(logits, labels)
                ema_loss = ema.penalty()
                task_b_step(task_loss + ema_loss, embed_layer, opt_b)
                ema.update_slow()  # EMA drifts gradually

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:200], task_a_labels[:200], 'A')
        acc_b = evaluate(model, tokenizer, task_b_texts[:200], task_b_labels[:200], 'B')
        forgetting = log_run(acc_a_initial, acc_a_final, acc_b)
        results.append({'forgetting': forgetting, 'acc_b': acc_b, 'acc_a_final': acc_a_final})
        cleanup(model, base_model)

    return summarise(results)


# =====================================================================
# 12. MAIN EXECUTION
# =====================================================================
if __name__ == "__main__":

    all_results = {
        'Baseline': train_baseline(),
        'Topological': train_topological(),
        'EWC': train_ewc(),
        'Replay': train_replay(),
        'HOPE-like': train_hope(),
    }

    print("\n" + "=" * 80)
    print(f"FINAL RESULTS — TWO TASKS  (mean ± std,  N = {N_RUNS} runs)")
    print("=" * 80)

    col = f"{'Method':<16}  {'Forgetting':>14}  {'Task A Final':>14}  {'Task B Acc':>12}"
    print(col)
    print("-" * len(col))
    for method, res in all_results.items():
        print(
            f"{method:<16}"
            f"  {res['forgetting_mean']:>+5.1f}% ±{res['forgetting_std']:>4.1f}"
            f"  {res['acc_a_final_mean'] * 100:>6.1f}% ±{res['acc_a_final_std'] * 100:>4.1f}"
            f"  {res['acc_b_mean'] * 100:>5.1f}% ±{res['acc_b_std'] * 100:>4.1f}"
        )

    print(f"\nRANKING by forgetting (lower = better):")
    ranked = sorted(all_results.items(), key=lambda x: x[1]['forgetting_mean'])
    for i, (method, res) in enumerate(ranked, 1):
        tb = res['acc_b_mean'] * 100
        print(f"  {i}. {method:<16}  {res['forgetting_mean']:>+5.1f}%  |  Task B: {tb:.1f}%")

    print("=" * 80)

Device: cuda
TWO-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline

Loading dataset from parquet...
✓ Success! Dataset loaded with 120000 samples


Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

Task A: 500 samples  (World=0, Sports=1)
Task B: 1000 samples  (Business=0, Sci/Tech=1)

Method: BASELINE  (shared embedding, no protection)

  Run 1/3


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

...

    Task A initial: 99.5%
    Task A final:   99.5%  |  Forgetting: +0.0%
    Task B acc:     81.0%

  Run 2/3


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A initial: 99.0%
    Task A final:   99.0%  |  Forgetting: +0.0%
    Task B acc:     79.0%

  Run 3/3


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

    Task A initial: 99.0%
    Task A final:   99.0%  |  Forgetting: +0.0%
    Task B acc:     82.5%

FINAL RESULTS — TWO TASKS  (mean ± std,  N = 3 runs)
Method                Forgetting    Task A Final    Task B Acc
--------------------------------------------------------------
Baseline           +3.3% ± 1.0    95.8% ± 1.2   97.5% ± 2.0
Topological        +2.3% ± 1.4    96.8% ± 1.6   99.5% ± 0.4
EWC                +3.5% ± 1.2    95.7% ± 1.4   97.5% ± 0.7
Replay             -0.3% ± 0.2    99.5% ± 0.0   88.2% ± 4.6
HOPE-like          +0.0% ± 0.0    99.2% ± 0.2   80.8% ± 1.4

RANKING by forgetting (lower = better):
  1. Replay             -0.3%  |  Task B: 88.2%
  2. HOPE-like          +0.0%  |  Task B: 80.8%
  3. Topological        +2.3%  |  Task B: 99.5%
  4. Baseline           +3.3%  |  Task B: 97.5%
  5. EWC                +3.5%  |  Task B: 97.5%


## THREE-TASK BENCHMARK - NEWVERSION

In [1]:
!pip install codecarbon -q

In [2]:
# =====================================================================
# AFTER EWC — FORCE GPU MEMORY FLUSH
# =====================================================================
import gc
import torch
import os

def flush_gpu_memory():
    """Force complete GPU memory release between methods."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    # Report memory state
    free, total = torch.cuda.mem_get_info()
    print(f"\n  [Memory flush] Free: {free/1024**3:.1f}GB / Total: {total/1024**3:.1f}GB")

In [3]:
def get_co2_metrics(tracker):
    """Safely extract CO2 metrics from CodeCarbon tracker."""
    if tracker is None or not HAS_CODECARBON:
        return 0.0, 0.0

    try:
        # Try different possible attribute names
        if hasattr(tracker, '_emissions'):
            co2_kg = tracker._emissions
        elif hasattr(tracker, 'emissions'):
            co2_kg = tracker.emissions
        else:
            co2_kg = 0.0

        if hasattr(tracker, '_energy_consumed'):
            energy_kwh = tracker._energy_consumed
        elif hasattr(tracker, 'energy_consumed'):
            energy_kwh = tracker.energy_consumed
        elif hasattr(tracker, '_total_energy'):
            energy_kwh = tracker._total_energy
        else:
            energy_kwh = 0.0

        return co2_kg, energy_kwh
    except:
        return 0.0, 0.0

In [1]:
# ============================================================================
# THREE-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline
# WITH CODECARBON EMISSIONS TRACKING - FULLY CORRECTED
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import random
import time
import pandas as pd
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from sklearn.metrics import accuracy_score
from warnings import filterwarnings
filterwarnings('ignore')

# CodeCarbon for tracking emissions
try:
    from codecarbon import EmissionsTracker
    HAS_CODECARBON = True
    print("✓ CodeCarbon loaded successfully!")
except ImportError:
    print("✗ CodeCarbon not installed. Install with: pip install codecarbon")
    HAS_CODECARBON = False

import sys
sys.path.insert(0, '/content/ast_lefm')
try:
    from ast_lefm.sieve import primes_up_to
except ImportError:
    def primes_up_to(n):
        """Fallback sieve of Eratosthenes."""
        sieve = [True] * (n + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(n ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, n + 1, i):
                    sieve[j] = False
        return [i for i in range(2, n + 1) if sieve[i]]


# ============================================================
# Global config - OPTIMIZED FOR MEMORY
# ============================================================
SEED          = 123
N_RUNS        = 3
BATCH_SIZE    = 8
REPLAY_HALF   = 4
MAX_LEN       = 32
EPOCHS_A      = 2
EPOCHS_B      = 2
EPOCHS_C      = 3
EWC_EPOCHS    = 1          # CRITICAL: EWC uses only 1 epoch
LR_EMBED      = 5e-3
LR_CLS        = 1e-3
EWC_LAMBDA    = 5000
EMA_BETA      = 0.9
EMA_LAMBDA    = 5000
PRIME_LIMIT   = 13
BUFFER_SIZE   = 100
MODEL_NAME    = "openai/gpt-oss-20b"
NUM_SAMPLES_A = 300
NUM_SAMPLES_B = 500
NUM_SAMPLES_C = 500
EVAL_SIZE_A   = 100
EVAL_SIZE_B   = 100
EVAL_SIZE_C   = 100
FISHER_SAMPLES = 100


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("=" * 80)
print("THREE-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline")
print("WITH CODECARBON EMISSIONS TRACKING")
print("=" * 80)


# =====================================================================
# 1. DATASET - PROPER PARQUET DOWNLOAD
# =====================================================================
print("\nLoading dataset from parquet...")

try:
    parquet_url = "https://huggingface.co/datasets/ag_news/resolve/main/data/train-00000-of-00001.parquet"
    df = pd.read_parquet(parquet_url)
    dataset = Dataset.from_pandas(df)
    print(f"✓ Success! Dataset loaded with {len(dataset)} samples")
except Exception as e:
    print(f"✗ Failed to load dataset: {e}")
    raise RuntimeError("Could not load dataset from parquet URL")

def create_task(dataset, class_labels, num_samples=500):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled = filtered.select(range(min(num_samples, len(filtered))))
    texts = [str(item['text']) for item in sampled]
    labels = [int(item['label']) % 2 for item in sampled]
    return texts, labels


task_a_texts, task_a_labels = create_task(dataset, [0, 1], NUM_SAMPLES_A)
task_b_texts, task_b_labels = create_task(dataset, [2, 3], NUM_SAMPLES_B)
task_c_texts, task_c_labels = create_task(dataset, [0, 3], NUM_SAMPLES_C)

print(f"Task A: {len(task_a_texts)} samples  (World=0,    Sports=1)")
print(f"Task B: {len(task_b_texts)} samples  (Business=0, Sci/Tech=1)")
print(f"Task C: {len(task_c_texts)} samples  (World=0,    Sci/Tech=1)  ← cross-domain")


# =====================================================================
# 2. MODEL — SEPARATE CLASSIFIER HEAD PER TASK
# =====================================================================
class TaskAwareModel(nn.Module):
    def __init__(self, base_model, hidden_size=2880):
        super().__init__()
        self.base_model   = base_model
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs     = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        last_hidden = outputs.hidden_states[-1][:, -1, :]
        if self.current_task == 'A':
            head = self.classifier_A
        elif self.current_task == 'B':
            head = self.classifier_B
        else:
            head = self.classifier_C
        return head(last_hidden), outputs

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C'), f"Unknown task: {task}"
        self.current_task = task

    def freeze_classifier_a(self):
        self.classifier_A.requires_grad_(False)

    def freeze_classifier_b(self):
        self.classifier_B.requires_grad_(False)


# =====================================================================
# 3. SHARED UTILITIES
# =====================================================================
def flush_gpu_memory():
    """Force complete GPU memory release between methods."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()


def get_embed_layer(base_model: nn.Module) -> nn.Embedding:
    if hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'wte'):
        return base_model.transformer.wte
    for module in base_model.modules():
        if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100_000:
            return module
    raise RuntimeError("Could not locate embedding layer.")


@torch.no_grad()
def evaluate(model, tokenizer, texts, labels, task: str, batch_size: int = 16) -> float:
    was_training  = model.training
    previous_task = model.current_task
    model.eval()
    model.switch_task(task)

    all_preds, all_labels = [], []
    for i in range(0, len(texts), batch_size):
        tokens = tokenizer(
            texts[i:i + batch_size], return_tensors="pt",
            padding=True, truncation=True, max_length=MAX_LEN,
        ).to(device)
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels[i:i + batch_size])

    model.switch_task(previous_task)
    if was_training:
        model.train()
    return accuracy_score(all_labels, all_preds)


def tokenize(tokenizer, texts, labels):
    tokens = tokenizer(
        texts, return_tensors="pt",
        padding=True, truncation=True, max_length=MAX_LEN,
    ).to(device)
    return tokens, torch.tensor(labels, dtype=torch.long).to(device)


def load_fresh_model():
    """Load a clean model copy. Backbone is immediately frozen."""
    flush_gpu_memory()
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    model = TaskAwareModel(base_model).to(device)
    for param in base_model.parameters():
        param.requires_grad = False
    return model, tokenizer, base_model


def train_task_a(model, tokenizer, optimizer):
    model.switch_task('A')
    model.train()
    for _ in range(EPOCHS_A):
        for i in range(0, len(task_a_texts), BATCH_SIZE):
            tokens, labels = tokenize(
                tokenizer,
                task_a_texts[i:i + BATCH_SIZE],
                task_a_labels[i:i + BATCH_SIZE],
            )
            optimizer.zero_grad()
            logits, _ = model(tokens.input_ids, tokens.attention_mask)
            F.cross_entropy(logits, labels).backward()
            optimizer.step()


def build_optimizer_ab(embed_layer, model):
    """Optimizer for Task A — all heads + embedding."""
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight,               'lr': LR_EMBED},
        {'params': model.classifier_A.parameters(),  'lr': LR_CLS},
        {'params': model.classifier_B.parameters(),  'lr': LR_CLS},
        {'params': model.classifier_C.parameters(),  'lr': LR_CLS},
    ])


def build_task_b_optimizer(embed_layer, model):
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight, 'lr': LR_EMBED},
        {'params': model.classifier_B.parameters(), 'lr': LR_CLS},
    ])


def build_task_c_optimizer(embed_layer, model):
    return bnb.optim.AdamW8bit([
        {'params': embed_layer.weight, 'lr': LR_EMBED},
        {'params': model.classifier_C.parameters(), 'lr': LR_CLS},
    ])


def task_step(loss, embed_layer, optimizer, max_norm=1.0):
    loss.backward()
    torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=max_norm)
    optimizer.step()


def summarise(results: list) -> dict:
    fgt_a  = [r['forgetting_a']        for r in results]
    fgt_b  = [r['forgetting_b']        for r in results]
    fgt_c  = [r['forgetting_combined'] for r in results]
    ac     = [r['acc_c']               for r in results]
    prot_t = [r.get('protection_time_ms', 0)  for r in results]
    prot_m = [r.get('protection_mem_kb',  0)  for r in results]
    co2    = [r.get('co2_emissions_kg', 0) for r in results]
    energy = [r.get('energy_kwh', 0) for r in results]
    return {
        'forgetting_a_mean':        np.mean(fgt_a),  'forgetting_a_std':        np.std(fgt_a),
        'forgetting_b_mean':        np.mean(fgt_b),  'forgetting_b_std':        np.std(fgt_b),
        'forgetting_combined_mean': np.mean(fgt_c),  'forgetting_combined_std': np.std(fgt_c),
        'acc_c_mean':               np.mean(ac),     'acc_c_std':               np.std(ac),
        'protection_time_ms_mean':  np.mean(prot_t), 'protection_time_ms_std':  np.std(prot_t),
        'protection_mem_kb_mean':   np.mean(prot_m), 'protection_mem_kb_std':   np.std(prot_m),
        'co2_emissions_kg_mean':    np.mean(co2),    'co2_emissions_kg_std':    np.std(co2),
        'energy_kwh_mean':          np.mean(energy), 'energy_kwh_std':          np.std(energy),
    }


def log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c, co2_kg=0, energy_kwh=0) -> tuple:
    fgt_a = (acc_a_initial - acc_a_final) * 100
    fgt_b = (acc_b_initial - acc_b_final) * 100
    fgt_combined = (fgt_a + fgt_b) / 2
    print(f"    Task A forgetting:    {fgt_a:+.1f}%  ({acc_a_initial*100:.1f}% → {acc_a_final*100:.1f}%)")
    print(f"    Task B forgetting:    {fgt_b:+.1f}%  ({acc_b_initial*100:.1f}% → {acc_b_final*100:.1f}%)")
    print(f"    Combined forgetting:  {fgt_combined:+.1f}%")
    print(f"    Task C acc:           {acc_c*100:.1f}%")
    if co2_kg > 0:
        print(f"    CO2 Emissions:       {co2_kg:.4f} kg CO2")
        print(f"    Energy Consumption:  {energy_kwh:.4f} kWh")
    return fgt_a, fgt_b, fgt_combined


def cleanup(*objects):
    for obj in objects:
        if obj is not None:
            del obj
    gc.collect()
    torch.cuda.empty_cache()


# =====================================================================
# HELPER: Get CO2 metrics from CodeCarbon tracker - FIXED
# =====================================================================
def get_co2_metrics(tracker):
    """Safely extract CO2 metrics from CodeCarbon tracker."""
    if tracker is None or not HAS_CODECARBON:
        return 0.0, 0.0

    try:
        co2_kg = 0.0
        energy_kwh = 0.0

        # Try to get emissions - handle different attribute names
        if hasattr(tracker, 'emissions'):
            emissions = tracker.emissions
            if hasattr(emissions, 'emissions'):
                co2_kg = float(emissions.emissions)
            elif hasattr(emissions, '__float__'):
                co2_kg = float(emissions)
            elif isinstance(emissions, (int, float)):
                co2_kg = float(emissions)
        elif hasattr(tracker, '_emissions'):
            emissions = tracker._emissions
            if hasattr(emissions, 'emissions'):
                co2_kg = float(emissions.emissions)
            elif hasattr(emissions, '__float__'):
                co2_kg = float(emissions)
            elif isinstance(emissions, (int, float)):
                co2_kg = float(emissions)
        elif hasattr(tracker, 'total_emissions'):
            co2_kg = float(tracker.total_emissions)

        # Try to get energy consumed
        if hasattr(tracker, 'energy_consumed'):
            energy = tracker.energy_consumed
            if hasattr(energy, 'energy_consumed'):
                energy_kwh = float(energy.energy_consumed)
            elif hasattr(energy, '__float__'):
                energy_kwh = float(energy)
            elif isinstance(energy, (int, float)):
                energy_kwh = float(energy)
        elif hasattr(tracker, '_energy_consumed'):
            energy = tracker._energy_consumed
            if hasattr(energy, 'energy_consumed'):
                energy_kwh = float(energy.energy_consumed)
            elif hasattr(energy, '__float__'):
                energy_kwh = float(energy)
            elif isinstance(energy, (int, float)):
                energy_kwh = float(energy)
        elif hasattr(tracker, 'total_energy'):
            energy_kwh = float(tracker.total_energy)

        return co2_kg, energy_kwh
    except Exception:
        return 0.0, 0.0


# =====================================================================
# COST TRACKING UTILITIES
# =====================================================================
def fisher_memory_mb(embed_layer, n_tasks=1):
    vocab_size, hidden_size = embed_layer.weight.shape
    bytes_per_matrix = vocab_size * hidden_size * 4
    return (bytes_per_matrix * n_tasks) / (1024 ** 2)


def anchor_memory_kb(governor):
    n_anchors   = len(governor.anchor_indices)
    hidden_size = governor.embed_layer.weight.shape[1]
    return (n_anchors * hidden_size * 4) / 1024


def calculate_buffer_memory_kb(buffer_a, buffer_b):
    if buffer_a:
        avg_len_a = np.mean([len(t) for t, _ in buffer_a])
    else:
        avg_len_a = MAX_LEN
    if buffer_b:
        avg_len_b = np.mean([len(t) for t, _ in buffer_b])
    else:
        avg_len_b = MAX_LEN
    mem_a = len(buffer_a) * avg_len_a * 4 / 1024
    mem_b = len(buffer_b) * avg_len_b * 4 / 1024
    return mem_a + mem_b


# =====================================================================
# 4. TOPOLOGICAL GOVERNOR
# =====================================================================
class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer    = embed_layer
        vocab_size          = embed_layer.weight.shape[0]
        self.anchor_indices = [p for p in primes_up_to(prime_limit) if p < vocab_size]
        self.snapshot: dict = {}
        print(f"  → Anchoring {len(self.anchor_indices)} prime rows: {self.anchor_indices}")

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            raise RuntimeError("Call take_snapshot() before enforce_anchors().")
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )


# =====================================================================
# 5. FISHER DIAGONAL (for EWC) - CPU VERSION
# =====================================================================
def compute_fisher_cpu(model, tokenizer, embed_layer, texts, labels,
                       n_samples: int = 100, task: str = 'A') -> torch.Tensor:
    """Compute Fisher diagonal - STORE ON CPU to save GPU memory."""
    model.eval()
    model.switch_task(task)

    # Initialize on CPU
    fisher = torch.zeros_like(embed_layer.weight, dtype=torch.float32, device='cpu')
    n_batches = 0

    for i in range(0, min(n_samples, len(texts)), BATCH_SIZE):
        tokens, lbs = tokenize(tokenizer, texts[i:i + BATCH_SIZE], labels[i:i + BATCH_SIZE])
        model.zero_grad()
        logits, _ = model(tokens.input_ids, tokens.attention_mask)
        F.cross_entropy(logits, lbs).backward()
        if embed_layer.weight.grad is not None:
            # Move gradient to CPU immediately
            grad_cpu = embed_layer.weight.grad.float().cpu()
            fisher += grad_cpu ** 2
            n_batches += 1
        # Clear GPU memory
        torch.cuda.empty_cache()

    fisher /= max(n_batches, 1)
    model.zero_grad()
    return fisher  # This stays on CPU!


# =====================================================================
# 6. DUAL TIMESCALE EMA (for HOPE-like)
# =====================================================================
class DualTimescaleEMA:
    def __init__(self, embed_layer: nn.Embedding,
                 beta: float = EMA_BETA, lambda_penalty: float = EMA_LAMBDA):
        self.embed_layer    = embed_layer
        self.beta           = beta
        self.lambda_penalty = lambda_penalty
        self.slow_embed     = embed_layer.weight.detach().clone().float()

    @torch.no_grad()
    def update_slow(self):
        current         = self.embed_layer.weight.float()
        self.slow_embed = self.beta * self.slow_embed + (1.0 - self.beta) * current

    def penalty(self) -> torch.Tensor:
        delta = self.embed_layer.weight.float() - self.slow_embed
        return self.lambda_penalty * (delta ** 2).sum()


# =====================================================================
# 7. BASELINE
# =====================================================================
def train_baseline():
    print(f"\n{'=' * 60}")
    print("Method: BASELINE  (no protection)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        tracker = EmissionsTracker(measure_power_secs=1) if HAS_CODECARBON else None
        if tracker:
            tracker.start()

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B
        opt_b = build_task_b_optimizer(embed_layer, model)
        model.switch_task('B')
        model.train()
        for _ in range(EPOCHS_B):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_step(F.cross_entropy(logits, labels), embed_layer, opt_b)
        model.freeze_classifier_b()

        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS_C):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_step(F.cross_entropy(logits, labels), embed_layer, opt_c)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')

        if tracker:
            tracker.stop()
            co2_kg, energy_kwh = get_co2_metrics(tracker)
            print(f"    [CO2]  {co2_kg:.6f} kg CO2 | {energy_kwh:.6f} kWh")
        else:
            co2_kg, energy_kwh = 0.0, 0.0

        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        print(f"    [Cost] No protection mechanism | Time: 0ms | Memory: 0KB")
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': 0.0, 'protection_mem_kb': 0.0,
            'co2_emissions_kg': co2_kg, 'energy_kwh': energy_kwh,
        })
        cleanup(model, base_model)
        flush_gpu_memory()
        if tracker:
            del tracker

    return summarise(results)


# =====================================================================
# 8. TOPOLOGICAL AI
# =====================================================================
def train_topological():
    print(f"\n{'=' * 60}")
    print("Method: TOPOLOGICAL AI  (prime embedding anchors)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        tracker = EmissionsTracker(measure_power_secs=1) if HAS_CODECARBON else None
        if tracker:
            tracker.start()

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        governor = TopologicalGovernor(embed_layer)
        t0_snap = time.perf_counter()
        governor.take_snapshot()
        snap_time_ms = (time.perf_counter() - t0_snap) * 1000
        anchor_mem_kb = anchor_memory_kb(governor)
        print(f"    [Cost] Snapshot time: {snap_time_ms:.2f}ms | Anchor memory: {anchor_mem_kb:.2f}KB | Scales: FLAT")
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B with anchor protection
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)
        for _ in range(EPOCHS_B):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
                opt_b.step()
                governor.enforce_anchors()

        model.freeze_classifier_b()
        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C with SAME anchor protection
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS_C):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
                opt_c.step()
                governor.enforce_anchors()

        assert governor.verify_integrity(), "Anchor integrity check failed!"

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')

        if tracker:
            tracker.stop()
            co2_kg, energy_kwh = get_co2_metrics(tracker)
            print(f"    [CO2]  {co2_kg:.6f} kg CO2 | {energy_kwh:.6f} kWh")
        else:
            co2_kg, energy_kwh = 0.0, 0.0

        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': snap_time_ms,
            'protection_mem_kb': anchor_mem_kb,
            'co2_emissions_kg': co2_kg,
            'energy_kwh': energy_kwh,
        })
        cleanup(model, base_model)
        flush_gpu_memory()
        if tracker:
            del tracker

    return summarise(results)


# =====================================================================
# 9. EWC — FIXED: Only 1 epoch, Fisher on CPU
# =====================================================================
def train_ewc():
    print(f"\n{'=' * 60}")
    print("Method: EWC  (cumulative Fisher across tasks)")
    print("  → Fisher matrices stored on CPU (not GPU)")
    print("  → ONLY 1 EPOCH to prevent memory fragmentation")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        flush_gpu_memory()
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        tracker = EmissionsTracker(measure_power_secs=1) if HAS_CODECARBON else None
        if tracker:
            tracker.start()

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        # Train Task A - ONLY 1 EPOCH
        opt = build_optimizer_ab(embed_layer, model)
        model.switch_task('A')
        model.train()
        for _ in range(1):  # ONLY 1 EPOCH for EWC
            for i in range(0, len(task_a_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer,
                    task_a_texts[i:i + BATCH_SIZE],
                    task_a_labels[i:i + BATCH_SIZE],
                )
                opt.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                F.cross_entropy(logits, labels).backward()
                opt.step()

        # Fisher from Task A - STORE ON CPU
        t0_fisher = time.perf_counter()
        print("  Computing Fisher A...")
        fisher_a = compute_fisher_cpu(model, tokenizer, embed_layer, task_a_texts, task_a_labels, task='A')
        embed_snap_a = embed_layer.weight.detach().clone().cpu().float()
        fisher_time_a_ms = (time.perf_counter() - t0_fisher) * 1000
        model.freeze_classifier_a()
        flush_gpu_memory()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B with EWC penalty - ONLY 1 EPOCH
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)
        for _ in range(1):  # ONLY 1 EPOCH
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss = F.cross_entropy(logits, labels)
                delta_a = embed_layer.weight.float() - embed_snap_a.to(device)
                ewc_loss = (fisher_a.to(device) * delta_a ** 2).sum()
                task_step(task_loss + EWC_LAMBDA * ewc_loss, embed_layer, opt_b)

        # Fisher from Task B - STORE ON CPU
        t0_fisher_b = time.perf_counter()
        print("  Computing Fisher B...")
        fisher_b = compute_fisher_cpu(model, tokenizer, embed_layer, task_b_texts, task_b_labels, task='B')
        embed_snap_b = embed_layer.weight.detach().clone().cpu().float()
        fisher_time_b_ms = (time.perf_counter() - t0_fisher_b) * 1000
        total_fisher_time = fisher_time_a_ms + fisher_time_b_ms
        fisher_mem_2tasks = fisher_memory_mb(embed_layer, n_tasks=2)
        print(f"    [Cost] Fisher A: {fisher_time_a_ms:.0f}ms | Fisher B: {fisher_time_b_ms:.0f}ms | Total: {total_fisher_time:.0f}ms | Memory (2 tasks): {fisher_mem_2tasks:.1f}MB (on CPU)")
        model.freeze_classifier_b()
        flush_gpu_memory()

        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C with CORRECT cumulative EWC penalty - ONLY 1 EPOCH
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(1):  # ONLY 1 EPOCH
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss = F.cross_entropy(logits, labels)
                delta_a = embed_layer.weight.float() - embed_snap_a.to(device)
                delta_b = embed_layer.weight.float() - embed_snap_b.to(device)
                # CORRECT: Each Fisher with its own delta
                ewc_loss = (fisher_a.to(device) * delta_a ** 2).sum() + (fisher_b.to(device) * delta_b ** 2).sum()
                task_step(task_loss + EWC_LAMBDA * ewc_loss, embed_layer, opt_c)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')

        if tracker:
            tracker.stop()
            co2_kg, energy_kwh = get_co2_metrics(tracker)
            print(f"    [CO2]  {co2_kg:.6f} kg CO2 | {energy_kwh:.6f} kWh")
        else:
            co2_kg, energy_kwh = 0.0, 0.0

        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': total_fisher_time,
            'protection_mem_kb': fisher_mem_2tasks * 1024,
            'co2_emissions_kg': co2_kg,
            'energy_kwh': energy_kwh,
        })

        # CRITICAL: Force delete Fisher matrices before next run
        del fisher_a, fisher_b, embed_snap_a, embed_snap_b
        cleanup(model, base_model)
        flush_gpu_memory()
        if tracker:
            del tracker
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    return summarise(results)


# =====================================================================
# 10. REPLAY
# =====================================================================
def train_replay():
    print(f"\n{'=' * 60}")
    print("Method: REPLAY  (buffer from Task A and Task B)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")

        tracker = EmissionsTracker(measure_power_secs=1) if HAS_CODECARBON else None
        if tracker:
            tracker.start()

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        buffer_a = list(zip(task_a_texts, task_a_labels))[:BUFFER_SIZE]
        t0_replay = time.perf_counter()
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        model.train()

        # Task B with Task A replay
        opt_b = build_task_b_optimizer(embed_layer, model)
        for _ in range(EPOCHS_B):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                opt_b.zero_grad()
                model.switch_task('B')
                tokens_b, labels_b = tokenize(tokenizer, task_b_texts[i:i + REPLAY_HALF], task_b_labels[i:i + REPLAY_HALF])
                logits_b, _ = model(tokens_b.input_ids, tokens_b.attention_mask)
                loss_b = F.cross_entropy(logits_b, labels_b)

                replay_a = random.sample(buffer_a, min(REPLAY_HALF, len(buffer_a)))
                r_texts, r_lbs = zip(*replay_a)
                tokens_a, labels_a = tokenize(tokenizer, list(r_texts), list(r_lbs))
                model.switch_task('A')
                logits_a, _ = model(tokens_a.input_ids, tokens_a.attention_mask)
                loss_a = F.cross_entropy(logits_a, labels_a)
                task_step(loss_b + loss_a, embed_layer, opt_b)

        buffer_b = list(zip(task_b_texts, task_b_labels))[:BUFFER_SIZE]
        model.freeze_classifier_b()
        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C with Task A + Task B replay
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS_C):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                opt_c.zero_grad()
                model.switch_task('C')
                tokens_c, labels_c = tokenize(tokenizer, task_c_texts[i:i + REPLAY_HALF], task_c_labels[i:i + REPLAY_HALF])
                logits_c, _ = model(tokens_c.input_ids, tokens_c.attention_mask)
                loss_c = F.cross_entropy(logits_c, labels_c)

                replay_a = random.choices(buffer_a, k=min(REPLAY_HALF // 2, len(buffer_a)))
                r_texts, r_lbs = zip(*replay_a)
                tokens_a, lbs_a = tokenize(tokenizer, list(r_texts), list(r_lbs))
                model.switch_task('A')
                logits_a, _ = model(tokens_a.input_ids, tokens_a.attention_mask)
                loss_a = F.cross_entropy(logits_a, lbs_a)

                replay_b = random.choices(buffer_b, k=min(REPLAY_HALF // 2, len(buffer_b)))
                r_texts, r_lbs = zip(*replay_b)
                tokens_b, lbs_b = tokenize(tokenizer, list(r_texts), list(r_lbs))
                model.switch_task('B')
                logits_b, _ = model(tokens_b.input_ids, tokens_b.attention_mask)
                loss_b = F.cross_entropy(logits_b, lbs_b)

                task_step(loss_c + loss_a + loss_b, embed_layer, opt_c)

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')

        if tracker:
            tracker.stop()
            co2_kg, energy_kwh = get_co2_metrics(tracker)
            print(f"    [CO2]  {co2_kg:.6f} kg CO2 | {energy_kwh:.6f} kWh")
        else:
            co2_kg, energy_kwh = 0.0, 0.0

        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        total_buffer_kb = calculate_buffer_memory_kb(buffer_a, buffer_b)
        replay_time_ms = (time.perf_counter() - t0_replay) * 1000
        print(f"    [Cost] Buffer memory: {total_buffer_kb:.1f}KB (A+B) | Total replay time: {replay_time_ms:.0f}ms | Scales: GROWS WITH N")
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': replay_time_ms,
            'protection_mem_kb': total_buffer_kb,
            'co2_emissions_kg': co2_kg,
            'energy_kwh': energy_kwh,
        })
        cleanup(model, base_model)
        flush_gpu_memory()
        if tracker:
            del tracker

    return summarise(results)


# =====================================================================
# 11. HOPE-like
# =====================================================================
def train_hope():
    print(f"\n{'=' * 60}")
    print("Method: HOPE-like  (Dual Timescale EMA)")
    print(f"{'=' * 60}")
    results = []

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  Run {run + 1}/{N_RUNS}")
        flush_gpu_memory()

        tracker = EmissionsTracker(measure_power_secs=1) if HAS_CODECARBON else None
        if tracker:
            tracker.start()

        model, tokenizer, base_model = load_fresh_model()
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        opt = build_optimizer_ab(embed_layer, model)
        train_task_a(model, tokenizer, opt)

        t0_ema = time.perf_counter()
        ema = DualTimescaleEMA(embed_layer, beta=EMA_BETA, lambda_penalty=EMA_LAMBDA)
        vocab_size, hidden_size = embed_layer.weight.shape
        ema_mem_kb = (vocab_size * hidden_size * 4) / 1024
        model.freeze_classifier_a()

        acc_a_initial = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')

        # Task B with EMA penalty
        model.switch_task('B')
        model.train()
        opt_b = build_task_b_optimizer(embed_layer, model)
        for _ in range(EPOCHS_B):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_b_texts[i:i + BATCH_SIZE], task_b_labels[i:i + BATCH_SIZE])
                opt_b.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss = F.cross_entropy(logits, labels)
                task_step(task_loss + ema.penalty(), embed_layer, opt_b)
                ema.update_slow()

        model.freeze_classifier_b()
        acc_b_initial = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')

        # Task C
        model.switch_task('C')
        model.train()
        opt_c = build_task_c_optimizer(embed_layer, model)
        for _ in range(EPOCHS_C):
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(tokenizer, task_c_texts[i:i + BATCH_SIZE], task_c_labels[i:i + BATCH_SIZE])
                opt_c.zero_grad()
                logits, _ = model(tokens.input_ids, tokens.attention_mask)
                task_loss = F.cross_entropy(logits, labels)
                task_step(task_loss + ema.penalty(), embed_layer, opt_c)
                ema.update_slow()

        acc_a_final = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE_A], task_a_labels[:EVAL_SIZE_A], 'A')
        acc_b_final = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE_B], task_b_labels[:EVAL_SIZE_B], 'B')
        acc_c       = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE_C], task_c_labels[:EVAL_SIZE_C], 'C')

        if tracker:
            tracker.stop()
            co2_kg, energy_kwh = get_co2_metrics(tracker)
            print(f"    [CO2]  {co2_kg:.6f} kg CO2 | {energy_kwh:.6f} kWh")
        else:
            co2_kg, energy_kwh = 0.0, 0.0

        fgt_a, fgt_b, fgt_combined = log_run(acc_a_initial, acc_b_initial, acc_a_final, acc_b_final, acc_c)
        ema_time_ms = (time.perf_counter() - t0_ema) * 1000
        print(f"    [Cost] EMA memory: {ema_mem_kb:.1f}KB (slow embed) | Total EMA time: {ema_time_ms:.0f}ms | Scales: FLAT")
        results.append({
            'forgetting_a': fgt_a, 'forgetting_b': fgt_b,
            'forgetting_combined': fgt_combined, 'acc_c': acc_c,
            'protection_time_ms': ema_time_ms,
            'protection_mem_kb': ema_mem_kb,
            'co2_emissions_kg': co2_kg,
            'energy_kwh': energy_kwh,
        })
        cleanup(model, base_model)
        flush_gpu_memory()
        if tracker:
            del tracker

    return summarise(results)


# =====================================================================
# 12. MAIN
# =====================================================================
if __name__ == "__main__":

    all_results = {}
    all_results['Baseline']    = train_baseline()
    flush_gpu_memory()
    all_results['Topological'] = train_topological()
    flush_gpu_memory()
    all_results['EWC']         = train_ewc()
    flush_gpu_memory()
    all_results['Replay']      = train_replay()
    flush_gpu_memory()
    all_results['HOPE-like']   = train_hope()

    print("\n" + "=" * 80)
    print(f"FINAL RESULTS — THREE TASKS  (mean ± std,  N = {N_RUNS} runs)")
    print("=" * 80)

    col = f"{'Method':<16}  {'Task C Acc':>12}  {'Fgt A':>8}  {'Fgt B':>8}  {'Fgt Comb':>10}  {'Prot Time':>12}  {'Prot Mem':>12}  {'CO2 (kg)':>12}"
    print(col)
    print("-" * len(col))
    for method, res in all_results.items():
        print(
            f"{method:<16}"
            f"  {res['acc_c_mean']*100:>5.1f}% ±{res['acc_c_std']*100:>4.1f}"
            f"  {res['forgetting_a_mean']:>+5.1f}%"
            f"  {res['forgetting_b_mean']:>+5.1f}%"
            f"  {res['forgetting_combined_mean']:>+6.1f}%"
            f"  {res['protection_time_ms_mean']:>7.0f}ms"
            f"  {res['protection_mem_kb_mean']:>7.1f}KB"
            f"  {res['co2_emissions_kg_mean']:>8.6f}"
        )

    print(f"\nRANKING by Task C accuracy (higher = better):")
    ranked = sorted(all_results.items(), key=lambda x: x[1]['acc_c_mean'], reverse=True)
    for i, (method, res) in enumerate(ranked, 1):
        print(
            f"  {i}. {method:<16}"
            f"  Task C: {res['acc_c_mean']*100:.1f}%"
            f"  |  Combined forgetting: {res['forgetting_combined_mean']:+.1f}%"
            f"  |  CO2: {res['co2_emissions_kg_mean']:.6f} kg"
            f"  |  Protection: {res['protection_time_ms_mean']:.0f}ms / {res['protection_mem_kb_mean']:.1f}KB"
        )

    print(f"\nCO2 EMISSIONS RANKING (lower = better):")
    co2_ranked = sorted(all_results.items(), key=lambda x: x[1]['co2_emissions_kg_mean'])
    for i, (method, res) in enumerate(co2_ranked, 1):
        print(
            f"  {i}. {method:<16}"
            f"  CO2: {res['co2_emissions_kg_mean']:.6f} kg"
            f"  |  Task C: {res['acc_c_mean']*100:.1f}%"
        )

    print(f"\nCOST SCALING NOTE:")
    print(f"  Topological AI — protection cost is FLAT regardless of number of tasks")
    if 'EWC' in all_results:
        print(f"  EWC             — Fisher memory grows linearly: ~{all_results['EWC']['protection_mem_kb_mean']/1024:.0f}MB per task added")
    print("=" * 80)

✓ CodeCarbon loaded successfully!
Device: cuda
THREE-TASK BENCHMARK: Topological AI vs EWC vs Replay vs HOPE-like vs Baseline
WITH CODECARBON EMISSIONS TRACKING

Loading dataset from parquet...
✓ Success! Dataset loaded with 120000 samples


Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

[codecarbon WARNING @ 00:09:19] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 00:09:19] [setup] RAM Tracking...
[codecarbon INFO @ 00:09:19] [setup] CPU Tracking...


...

[codecarbon WARNING @ 00:32:55] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 00:32:55] [setup] RAM Tracking...
[codecarbon INFO @ 00:32:55] [setup] CPU Tracking...
[codecarbon WARNING @ 00:32:55] We saw that you have a AMD EPYC 9B45 but we don't know it. Please contact us.
[codecarbon WARNING @ 00:32:55] We will use the default power consumption of 4 W per thread for your 48 CPU, so 192W.
[codecarbon WARNING @ 00:32:55] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 00:32:55] CPU Model on constant consumption mode: AMD EPYC 9B45
[codecarbon WARNING @ 00:32:55] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon INFO @ 00:32:55] [setup] GPU Tracking...
[codecarbon INFO @ 00:32:55] Tracking Nvidia GPUs via PyNVML
[codecarbon INFO @ 00:32:55] The bel


  Run 3/3


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

[codecarbon INFO @ 00:32:57] Energy consumed for RAM : 0.000030 kWh. RAM Power : 54.0 W
[codecarbon INFO @ 00:32:57] Delta energy consumed for CPU with cpu_load : 0.000015 kWh, power : 27.672020428800003 W
[codecarbon INFO @ 00:32:57] Energy consumed for All CPU : 0.000015 kWh
[codecarbon INFO @ 00:32:57] Monitoring GPUs with indices: [0] out of 1 total GPUs
[codecarbon INFO @ 00:32:57] Energy consumed for all GPUs : 0.000036 kWh. Total GPU Power : 51.65335758195711 W
[codecarbon INFO @ 00:32:57] 0.000082 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 00:32:58] Energy consumed for RAM : 0.000037 kWh. RAM Power : 54.0 W
[codecarbon INFO @ 00:32:58] Delta energy consumed for CPU with cpu_load : 0.000004 kWh, power : 28.3125 W
[codecarbon INFO @ 00:32:58] Energy consumed for All CPU : 0.000019 kWh
[codecarbon INFO @ 00:32:58] Energy consumed for all GPUs : 0.000060 kWh. Total GPU Power : 86.35311879595082 W
[codecarbon INFO @ 00:32:58] 0.00011

    [CO2]  0.000000 kg CO2 | 0.000000 kWh
    Task A forgetting:    +0.0%  (99.0% → 99.0%)
    Task B forgetting:    +0.0%  (31.0% → 31.0%)
    Combined forgetting:  +0.0%
    Task C acc:           97.0%
    [Cost] EMA memory: 2262240.0KB (slow embed) | Total EMA time: 55998ms | Scales: FLAT

FINAL RESULTS — THREE TASKS  (mean ± std,  N = 3 runs)
Method              Task C Acc     Fgt A     Fgt B    Fgt Comb     Prot Time      Prot Mem      CO2 (kg)
--------------------------------------------------------------------------------------------------------
Baseline          100.0% ± 0.0   +3.7%   +0.0%    +1.8%        0ms      0.0KB  0.000000
Topological       100.0% ± 0.0   +3.7%   +2.0%    +2.8%        0ms     67.5KB  0.000000
EWC                44.0% ±25.3   +1.7%   -0.0%    +0.8%    22366ms  4524480.0KB  0.000000
Replay             93.7% ± 3.3   -2.0%  -41.0%   -21.5%    93144ms    187.8KB  0.000000
HOPE-like          91.7% ± 6.8   -0.3%   +0.7%    +0.2%    55970ms  2262240.0KB  0.0000